# Week 100, Day 3: Mock loop

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w100d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from decimal import Decimal, ROUND_HALF_UP
print(f"{2.675:.2f}", Decimal("2.675").quantize(Decimal("0.01"), rounding=ROUND_HALF_UP))

*Interactive figure on the website: Two and a half hours for the whole loop..*

### Exercise: Coding 1: Merging sorted logs

Several servers each give a list of `(timestamp, message)` sorted by time. Write `merge_logs(logs)`: one list sorted by time (ties keep the order of the servers given), in O(N log k) with a heap.

In [ ]:
import heapq

def merge_logs(logs):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, time\n\ndef test_merge():\n    \"a heap of the next item from each\"\n    a = [(1, \"a1\"), (4, \"a4\"), (9, \"a9\")]\n    b = [(2, \"b2\"), (4, \"b4\"), (10, \"b10\")]\n    assert merge_logs([a, b]) == [(1, \"a1\"), (2, \"b2\"), (4, \"a4\"), (4, \"b4\"), (9, \"a9\"), (10, \"b10\")]\n    assert merge_logs([]) == [] and merge_logs([[], [(3, \"x\")]]) == [(3, \"x\")]\n\ndef test_fast():\n    \"100 servers, 2,000 lines each\"\n    rng = random.Random(1)\n    logs = [sorted((rng.randint(0, 10**6), f\"s{s}\") for _ in range(2000)) for s in range(100)]\n    start = time.perf_counter()\n    merged = merge_logs(logs)\n    assert len(merged) == 200_000 and all(merged[i][0] <= merged[i + 1][0] for i in range(len(merged) - 1))\n    assert time.perf_counter() - start < 3")

### Exercise: Coding 2: Fog over the hills

On a map, `"F"` cells are foggy, `"."` cells clear and `"#"` cells rock; each minute fog spreads to clear cells next to it (up, down, left, right). Write `fog_minutes(grid)`: the minutes until no clear cell is left, or `-1` if some clear cell is never reached.

In [ ]:
from collections import deque

def fog_minutes(grid):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_fog():\n    \"BFS from every source at once\"\n    assert fog_minutes([\"F..\", \"..#\", \"#..\"]) == 4 and fog_minutes([\"F#.\", \"##.\"]) == -1\n    assert fog_minutes([\"FF\", \"FF\"]) == 0 and fog_minutes([\"#\"]) == 0 and fog_minutes([\"F...F\"]) == 2\n    big = [\"F\" + \".\" * 299] + [\".\" * 300] * 299\n    assert fog_minutes(big) == 598")

### Exercise: Design: how big is the search index?

Waymark wants full-text search over walk descriptions. Write `index_size(docs, words_per_doc, distinct_share, bytes_per_posting, bytes_per_term, vocabulary)`: an inverted index stores, for every distinct word in each document, one **posting** (`docs × words_per_doc × distinct_share` postings), plus a dictionary of `vocabulary` terms. Return a dict with `postings` (rounded to a whole number), `posting_gb` and `dictionary_mb` (1 decimal place, GB = 10^9 bytes, MB = 10^6 bytes), and `total_gb` (1 decimal place).

In [ ]:
def index_size(docs, words_per_doc, distinct_share, bytes_per_posting, bytes_per_term, vocabulary):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_index():\n    \"postings dominate\"\n    assert index_size(10_000_000, 200, 0.6, 8, 40, 2_000_000) == {\"postings\": 1_200_000_000, \"posting_gb\": 9.6, \"dictionary_mb\": 80.0, \"total_gb\": 9.7}\n    assert index_size(1000, 10, 1.0, 4, 10, 100)[\"postings\"] == 10_000")

### Bug hunt: A pace rounded to nothing

Floor division turns 100 minutes over 6 km into `16` minutes a kilometre. Use true division and round to 1 place so it prints `16.7`.

In [ ]:
minutes, km = 100, 6
print(minutes // km)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"true division\"\n    expect_output(\"16.7\")")

### Bug hunt: Half a penny the wrong way

Formatting the float 2.675 rounds down (it's stored as 2.67499...), so a price of £2.675 shows as `2.67`. Use `Decimal` with `ROUND_HALF_UP` from the string so this prints `2.68`.

In [ ]:
from decimal import Decimal, ROUND_HALF_UP

price = 2.675
print(f"{price:.2f}")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"money is Decimal\"\n    expect_output(\"2.68\")")

### Exercise: Debugging: the last chunk disappeared

`chunks(items, n)` should split a list into pieces of `n`, with a shorter last piece if needed, but it drops the last partial piece. Fix it; raise `ValueError` if `n < 1`.

In [ ]:
def chunks(items, n):
    return [items[i:i + n] for i in range(0, len(items) - n + 1, n)]

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_chunks():\n    \"nothing lost\"\n    assert chunks(list(range(7)), 3) == [[0, 1, 2], [3, 4, 5], [6]] and chunks([], 2) == [] and chunks([1, 2], 2) == [[1, 2]]\n    try:\n        chunks([1], 0)\n    except ValueError:\n        pass\n    else:\n        raise AssertionError(\"n = 0\")")

### Exercise: Deep dive: versions that sort

Write a class `Version` decorated with `functools.total_ordering`: `Version("1.10.0")` parses `MAJOR.MINOR.PATCH` (raise `ValueError` otherwise), compares and sorts numerically, is equal to another `Version` with the same numbers (and `NotImplemented` against other types), hashes consistently, and shows as `Version('1.10.0')`. Define only `__eq__` and `__lt__` for the comparisons.

In [ ]:
import functools

@functools.total_ordering
class Version:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_versions():\n    \"two methods, six comparisons\"\n    v = [Version(s) for s in [\"1.10.0\", \"1.9.9\", \"2.0.0\", \"1.9.10\"]]\n    assert [str(x.parts) for x in sorted(v)] == [\"(1, 9, 9)\", \"(1, 9, 10)\", \"(1, 10, 0)\", \"(2, 0, 0)\"]\n    assert Version(\"1.10.0\") > Version(\"1.9.0\") and Version(\"1.2.3\") >= Version(\"1.2.3\") and Version(\"1.2.3\") != Version(\"1.2.4\")\n    assert len({Version(\"1.0.0\"), Version(\"1.0.0\")}) == 1 and repr(Version(\"1.10.0\")) == \"Version('1.10.0')\"\n    assert (Version(\"1.0.0\") == \"1.0.0\") is False\n    for bad in [\"1.0\", \"a.b.c\", \"1.0.0.0\"]:\n        try:\n            Version(bad)\n        except ValueError:\n            pass\n        else:\n            raise AssertionError(bad)\n    assert \"__gt__\" not in Version.__dict__ or Version.__gt__ is not None")

> **Behavioural: a time you helped someone else succeed.** Two minutes, aloud: "Tell me about a time you helped someone else do better work." A review you gave in Week 96 is a good story.

### Puzzle: Spreading from every source

Arrange the core of `fog_minutes`. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # fog_minutes(["F.F"])
q = ___   # chunks([1, 2, 3], 2) after the fix
r = ___   # Version("1.10.0") < Version("1.9.0")
s = ___   # the two comparison methods total_ordering needs you to write, as a list
t = ___   # f"{0.125:.2f}" (as Python prints it)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 1 and q == [[1, 2], [3]] and r is False, \"p, q and r\"\n    assert sorted(s) == [\"__eq__\", \"__lt__\"] and t == \"0.12\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

`2.675` can't be stored exactly as a binary float (it's really 2.67499999...), so it rounds down. For money, start from `Decimal` strings and choose the rounding rule explicitly. It's one of today's debugging rounds.

## The loop

### Coding round

### Design round

### Debugging round

### Python deep-dive round

### Behavioural round

## Quiz

## Scoring yourself

Score each round as last week and add the scores to your log: Saturday's final debrief uses all three of this week's loops.

## Before you go

Tomorrow is the second final loop.